# Notebook 03: Fine-Tuning Runs

**Goal:** Fine-tune on Python code (primary) and TinyStories prose (mandatory control).

**Outputs:** `checkpoints/code_seed*/`, `checkpoints/prose_seed*/`

**Runtime:** ~80 minutes per condition on Colab T4 GPU.

> **Note:** Both conditions are required. The prose control is mandatory for any domain-shift claim.

In [ ]:
import os
import sys
import numpy as np, torch, matplotlib.pyplot as plt
from pathlib import Path

# Define repository information
repo_name = "Mechanistic-Interpretability-Study-Induction-Circuit-Stability-Under-Fine-Tuning"
repo_url = "https://github.com/Mattral/Mechanistic-Interpretability-Study-Induction-Circuit-Stability-Under-Fine-Tuning"
repo_path = f"/content/{repo_name}"  # Standard Colab clone location

# Clone the repository if it doesn't exist
if not os.path.exists(repo_path):
    print(f"Cloning {repo_url} to {repo_path}...")
    !git clone {repo_url} {repo_path}

# Change current working directory to the repository root
# This allows relative imports (like 'src.model...') to work correctly
if os.getcwd() != repo_path:
    print(f"Changing current directory to {repo_path}")
    os.chdir(repo_path)

# Add the current directory (repo root) to sys.path if not already there
# This ensures 'src' is discoverable for imports.
if '.' not in sys.path:
    sys.path.insert(0, '.')

# Install project dependencies from requirements.txt
# This ensures all necessary libraries, including transformer_lens and transformers,
# are installed with the versions specified by the project.
print(f"Installing dependencies from {repo_path}/requirements.txt...")
!pip install -r requirements.txt

# Original imports
from src.model.config import ModelConfig, EvalConfig
from src.model.train import load_pretrained_model, set_global_seed
from src.circuits.patching import compute_circuit_attribution, get_circuit_heads
from src.viz.circuit_diagram import plot_circuit_diagram, plot_attribution_heatmap

set_global_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

Please restart the runtime immediately after this cell finishes execution (Runtime -> Restart runtime... or Ctrl/Cmd + M .).

In [ ]:
# After restarting the runtime, re-run this cell to continue with the imports and model setup.
import numpy as np, torch, matplotlib.pyplot as plt
from pathlib import Path

# The current working directory should already be set to the repo root from the previous cell.
# Add the current directory (repo root) to sys.path if not already there, in case of a fresh restart.
import sys
import os

repo_name = "Mechanistic-Interpretability-Study-Induction-Circuit-Stability-Under-Fine-Tuning"
repo_path = f"/content/{repo_name}"

if os.getcwd() != repo_path:
    print(f"Changing current directory to {repo_path}")
    os.chdir(repo_path)

if '.' not in sys.path:
    sys.path.insert(0, '.')

from src.model.config import ModelConfig, EvalConfig
from src.model.train import load_pretrained_model, set_global_seed
from src.circuits.patching import compute_circuit_attribution, get_circuit_heads
from src.viz.circuit_diagram import plot_circuit_diagram, plot_attribution_heatmap

set_global_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

In [ ]:
import sys; sys.path.insert(0, '..')
import torch
from src.model.config import ModelConfig, TrainConfig
from src.model.finetune import run_finetuning
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
model_config = ModelConfig()

In [ ]:
# --- Code fine-tuning (primary condition, 3 seeds) ---
for seed in [42, 123, 7]:
    print(f'\n=== Code, seed={seed} ===')
    cfg = TrainConfig(seed=seed, checkpoint_dir='../checkpoints', results_dir='../experiments/results')
    history = run_finetuning(model_config, cfg, run_name=f'code_seed{seed}', device=device, prose_control=False)
    print(f'Done: {len(history)} checkpoints')

In [ ]:
# --- Prose control (mandatory, 3 seeds) ---
for seed in [42, 123, 7]:
    print(f'\n=== Prose, seed={seed} ===')
    cfg = TrainConfig(seed=seed, checkpoint_dir='../checkpoints', results_dir='../experiments/results')
    history = run_finetuning(model_config, cfg, run_name=f'prose_seed{seed}', device=device, prose_control=True)
    print(f'Done: {len(history)} checkpoints')